# 구조 파이프라인 조종석 (Structure Pipeline)

**`VISION_final_modeling.md`의 실행장** — 라벨 검증 → 기준선 분석 → 타깃 증강(트랙 1) → 미니 A/B(트랙 2) → 게이트 판정 → 제출.
단기 일정·게이트 기준은 `PLAN_post_labeling.md`, 과거 실험 재현은 기존 `Train_Experiments.ipynb` 유지.

**주의 (기존 함정 그대로 적용)**
- GPU는 하나 = 한 번에 한 작업. **gemma 라벨링이 도는 동안 학습 금지** (라벨링도 GPU 점유)
- 본학습은 백그라운드 프로세스 — 커널이 죽어도 학습은 계속, 밤샘은 전원 + 덮개 열기
- 판정은 항상 **acc_shuffled** (무작위 4.2%, ±4%p는 노이즈) | 프롬프트-어댑터 세트 원칙
- 이 노트북 실행 중 .ipynb 파일 외부 수정 금지 (scripts/*.py 수정은 다음 프로세스부터 적용)

In [ ]:
# ⓪ 상태 보드 — 라벨링 진행·GPU·신규 라벨 수 (매 세션 처음 실행)
import subprocess, glob, os
print(subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
prog = "./outputs/gemma_labels/progress.txt"
if os.path.exists(prog):
    print("라벨링:", open(prog, encoding="utf-8").read().strip())
parts = sorted(glob.glob("./outputs/gemma_labels/parts/part_*.jsonl"))
n_new = sum(sum(1 for _ in open(p, encoding="utf-8")) for p in parts)
print(f"gemma 신규분: parts {len(parts)}개 파일 {n_new}건 (+ 기존 labels.jsonl 902건)")

In [ ]:
# ① gemma 라벨 통합 로드 + 품질 점검 (labels.jsonl + parts/*.jsonl 자동 병합)
import sys; sys.path.insert(0, "scripts")
import pandas as pd
from structure_features import load_gemma_labels, camera_regex

g = load_gemma_labels()
g["camera_re"] = g.sentence.map(camera_regex)
print(f"camera: gemma {g.camera.mean():.1%} vs 정규식 {g.camera_re.mean():.1%} "
      f"| 일치율 {(g.camera == g.camera_re).mean():.1%} (7/17 기준 92.1%)")
print(f"viewer: {g.viewer.mean():.1%} — holdout 실측은 역방향(-6.8%p, n=19)이라 camera와 병합 금지")
print("n_events 분포:"); print(g.n_events.value_counts().sort_index().to_string())

In [ ]:
# ② 기준선 분석 — holdout x exp07 축별 정확도 → 정확도 낮은 구조 조건(약점)이 증강 가중(④)의 근거
#    ⚠️ 규정(PROJECT_SETUP §4.3): "평가 데이터 특성 분석 후 학습"은 실격 — test 분포는 여기서 절대 안 씀
#    판정은 섞인 샘플만 (identity 착시 제거)
pred = pd.read_csv("./outputs/preds/Qwen3-VL-2B-Instruct_exp07_aug2_full.csv")
pred = pred[pred["gt"] != "[1, 2, 3, 4]"]   # .gt는 pandas 메서드와 충돌 — 반드시 ["gt"]
m = pred.merge(g[g.split == "holdout"][["Id", "camera", "viewer", "n_events", "n_subj", "n_markers"]],
               on="Id", how="left")

for col in ["camera", "viewer", "n_events", "n_subj", "n_markers"]:
    t = m.dropna(subset=[col]).groupby(col).correct.agg(["mean", "size"])
    print(f"\n[{col}]")
    print((t["mean"].map("{:.1%}".format) + "  (n=" + t["size"].astype(str) + ")").to_string())

print("\n(test 분포 분석 셀은 규정 위반 소지로 제거 — 기준선은 위 holdout 약점만 사용)")

In [ ]:
# ③ 실험 레지스트리 — 기본값과 다른 것만 명시. 실험 추가 = 한 줄
DEFAULTS = dict(model="./models/Qwen3-VL-2B-Instruct", load_4bit=False, aug_mult=2, lr=1e-4,
                epochs=1, max_hours=0, prompt="v1_list", script="scripts/train.py",
                extra="", eval_extra="")

EXPERIMENTS = {
    # --- 트랙 1: 타깃 증강 (VISION 1단계) ---
    "exp15_camaug": dict(extra="--aug-weights ./outputs/aug_weights_exp15.csv --snapshot-steps 150"),
    # --- 트랙 2: 미니 A/B (라벨 완성 후 활성화 — 남은 구현은 맨 아래 셀 참조) ---
    # "mini_hint_aug2":      dict(prompt="v6_hint", extra="--max-samples 1000 --max-steps 300"),
    # "mini_gemma_cot_aug2": dict(script="scripts/train_cot.py", eval_extra="--max-new-tokens 512",
    #                             extra="--max-samples 1000 --max-steps 300"),
}

def cfg(name): return {**DEFAULTS, **EXPERIMENTS[name]}

def train_cmd(name, smoke=False):
    c = cfg(name)
    run = name + ("_smoke" if smoke else "")
    cmd = (f"python {c['script']} --run-name {run} --model {c['model']} "
           f"--aug-mult {c['aug_mult']} --lr {c['lr']} --epochs {c['epochs']} "
           f"--max-hours {c['max_hours']} --prompt {c['prompt']} {c['extra']}").strip()
    if c["load_4bit"]: cmd += " --load-4bit"
    if smoke: cmd += " --max-samples 12 --grad-accum 4 --max-steps 5"
    return cmd

def eval_cmd(name):
    c = cfg(name)
    cmd = (f"python scripts/eval_zero_shot.py --model {c['model']} "
           f"--adapter ./outputs/runs/{name}/adapter --prompt {c['prompt']} {c['eval_extra']}").strip()
    if c["load_4bit"]: cmd += " --load-4bit"
    return cmd

NAME = "exp15_camaug"
print(f"대상: {NAME}\n  학습: {train_cmd(NAME)}\n  평가: {eval_cmd(NAME)}")

In [ ]:
# ④ 데이터 제작 — 증강 가중 CSV + CLIP 피처 수령 인터페이스
from structure_features import build_feature_table, make_aug_weights, load_clip_features
from train import load_excluded_ids

train = pd.read_csv("./snuaichallenge_data/train.csv")
train = train[~train.Id.isin(load_excluded_ids())].reset_index(drop=True)
ft = build_feature_table(train)

# exp15: 카메라X x4 / 카메라O x2 (정규식 축 — 라벨링 진행과 무관, 언제든 재생성 가능)
make_aug_weights(ft, [(lambda r: not r.camera_re, 4)], 2, "./outputs/aug_weights_exp15.csv")

# CLIP 피처 (팀원 산출물 인터페이스): train은 보유, test는 도착하면 주석 해제 → 스키마 자동 검증
clip_train = load_clip_features()                                  # ./snu_clip_features.csv
# clip_test = load_clip_features("./snu_clip_features_test.csv")   # test 819 도착 후

In [ ]:
# ⑤ 스모크 (~3분) — 코드/설정 변경 시 본학습 전 필수. peak VRAM 7.5GB 초과 시 중단하고 상의
!{train_cmd(NAME, smoke=True)}

In [ ]:
# ⑥ 본학습 시작 — 백그라운드 (커널을 막지 않음, 커널이 죽어도 학습은 계속)
import subprocess, os
os.makedirs(f"./outputs/runs/{NAME}", exist_ok=True)
console = open(f"./outputs/runs/{NAME}/console.log", "w", encoding="utf-8")
proc = subprocess.Popen(train_cmd(NAME), shell=True, stdout=console, stderr=subprocess.STDOUT,
                        env={**os.environ, "PYTHONIOENCODING": "utf-8"})
print(f"학습 시작: PID {proc.pid} | 콘솔 로그: outputs/runs/{NAME}/console.log")
print("중단하려면: 새 셀에서 proc.kill() (또는 작업관리자에서 위 PID 트리 종료)")

In [ ]:
# ⑦ 진행 게이지 — 30초마다 갱신, 완료 시 자동으로 다음 셀(평가)로
import sys, time
from IPython.display import clear_output
sys.path.insert(0, "scripts")
from watch_train import read_status, render

run_dir = f"./outputs/runs/{NAME}"
p = globals().get("proc")
while True:
    try:
        c, last = read_status(run_dir)
        line = render(c, last)
        done = (p is not None and p.poll() is not None) or \
               (last is not None and last.opt_step >= c["total_opt_steps"])
    except FileNotFoundError:
        line, done = "시작 대기 중... (모델 로드에 1~2분)", False
    clear_output(wait=True)
    print(time.strftime("%H:%M:%S"), line)
    if done:
        tail = open(f"{run_dir}/console.log", encoding="utf-8", errors="replace").read().splitlines()
        print("\n학습 종료 — 콘솔 로그 마지막:\n" + "\n".join(tail[-3:]))
        break
    time.sleep(30)

In [ ]:
# ⑧ holdout 300 평가 — experiments.csv에 자동 누적
!{eval_cmd(NAME)}

In [ ]:
# ⑨ 게이트 판정 (PLAN_post_labeling.md 합의 기준) + 세그먼트 분해
import glob as _glob

GATES = {  # name: (기준 실험, 기준 acc_shuffled, 요구 격차)
    "exp15_camaug":        ("exp07_aug2_full", 0.4841, 0.02),
    "mini_hint_aug2":      ("mini_v1_aug2",    0.1548, 0.04),
    "mini_gemma_cot_aug2": ("mini_v1_aug2",    0.1548, 0.04),
}
exp = pd.read_csv("./outputs/experiments.csv")
rows = exp[exp.model_id.str.contains(NAME, na=False)]
assert len(rows), f"{NAME} 평가 기록 없음 — ⑧을 먼저 실행"
acc = rows.iloc[-1].acc_shuffled
base_name, base, need = GATES[NAME]
diff = acc - base
verdict = "승 — 확전/주력 후보" if diff >= need else ("보류 (노이즈 범위)" if diff > -need else "기각")
print(f"{NAME} {acc:.1%} vs {base_name} {base:.1%} = {diff:+.1%} (게이트 {need:+.0%}) → {verdict}")

# 세그먼트 분해 — 타깃(카메라X 등)이 '실제로' 올랐는지 확인 (총점 동률이어도 구성이 다를 수 있음: exp14 전례)
def seg(pred_path, label):
    p = pd.read_csv(pred_path)
    p = p[p["gt"] != "[1, 2, 3, 4]"]
    if "Sentence" not in p.columns:
        p = p.merge(pd.read_csv("./splits/holdout_300.csv")[["Id", "Sentence"]], on="Id")
    p["camera_re"] = p.Sentence.map(camera_regex)
    t = p.groupby("camera_re").correct.agg(["mean", "size"])
    print(f"[{label:20s}] " + " | ".join(
        f"카메라{'O' if k else 'X'} {v['mean']:.1%} (n={int(v['size'])})" for k, v in t.iterrows()))

mine = sorted(_glob.glob(f"./outputs/preds/*{NAME}*.csv"))
if mine:
    seg(mine[-1], NAME)
seg("./outputs/preds/Qwen3-VL-2B-Instruct_exp07_aug2_full.csv", "exp07 기준")

In [ ]:
# ⑩ 제출 생성 — ⑨가 '승'일 때만 주석 해제 (팀 전체 1일 2회 제한, 학습·라벨링과 GPU 동시 사용 금지)
# import prompt_lab as lab, prompts as prompt_registry
# model, processor = lab.load_model(adapter=f"./outputs/runs/{NAME}/adapter")
# test_df = lab.load_testset()
# text_fn = lambda row: prompt_registry.build_user_text(cfg(NAME)["prompt"], row.Sentence)
# lab.make_submission(NAME, text_fn, model, processor, test_df, max_new_tokens=32)
print("게이트 승자 확정 후 주석 해제해서 실행")

---
## 트랙 2 활성화까지 남은 구현 (분담: 라벨 파싱 = 우리, 이미지 수치 = 팀원)

| 조각 | 상태 |
|---|---|
| `scripts/structure_features.py` (라벨 병합·정규식·가중 생성) | 완료 |
| `train.py --aug-weights` (행별 가변 증강) | 완료 |
| `scripts/make_hint_data.py` (gemma 라벨 → 힌트 학습셋, A트랙) | 미구현 |
| `prompts.py`에 `v6_hint` (힌트 슬롯 프롬프트) | 미구현 |
| `train_cot.py`의 타깃 소스를 gemma events로 교체 (B트랙) | 미구현 |
| test 819 gemma 라벨 + test CLIP 피처 — **추론 전처리 용도로만** (학습 설계 사용 = 실격) | 라벨=우리 / CLIP=팀원 |
| 규정 (7/17 확인): 추론 시 전처리는 허용 판단, test 분석→학습은 실격 조항 | gemma4 가중치 공개일(≤2026-05-31)만 확인 남음 |